In [ ]:
#!/usr/bin/env python3
"""Simulate SBS96 mutation count matrices from COSMIC signatures, with ground truth.

Generative model, per sample j:
    active set A_j      : "always-on" signatures + each other signature with prob p_active
    proportions h_j     : Dirichlet(alpha) over A_j (zero elsewhere)
    burden B_j          : LogNormal(log(median_burden), burden_sigma)
    signature W         : COSMIC columns, optionally perturbed  W'_k ~ Dirichlet(conc * W_k)
    expected counts     : mu_j = B_j * W' h_j
    observed counts     : X_j ~ Poisson(mu_j)  or  NegBin(mean mu_j, dispersion r)

Outputs (in --out):
    counts.tsv              96 x N  observed counts X (SigProfiler-style, Type column first)
    W_true.tsv              96 x K  signatures actually used to generate (after perturbation)
    W_cosmic.tsv            96 x K  the unperturbed COSMIC columns
    H_true_counts.tsv       K x N   mutations attributed to each signature (B_j * h_j)
    H_true_proportions.tsv  K x N   h_j
    params.json             all settings + seed, so the dataset can be regenerated
"""
import argparse
import json
import os

import numpy as np
import pandas as pd

HERE = os.path.dirname(os.path.abspath(__file__))
DEFAULT_COSMIC = os.path.join(HERE, "data", "COSMIC_v3.4_SBS_GRCh37.txt")

# COSMIC flags these as possible sequencing artefacts; excluded from random draws by default.
ARTEFACTS = {"SBS27", "SBS43", "SBS45", "SBS46", "SBS47", "SBS48", "SBS49", "SBS50", "SBS51",
             "SBS52", "SBS53", "SBS54", "SBS55", "SBS56", "SBS57", "SBS58", "SBS59", "SBS60",
             "SBS95"}

# Hand-picked sets for common scenarios (loosely based on PCAWG tissue activity).
PRESETS = {
    "lung":     ["SBS1", "SBS2", "SBS4", "SBS5", "SBS13", "SBS40a"],
    "melanoma": ["SBS1", "SBS5", "SBS7a", "SBS7b", "SBS7c", "SBS7d", "SBS38"],
    "breast":   ["SBS1", "SBS2", "SBS3", "SBS5", "SBS8", "SBS13", "SBS18"],
    "colon":    ["SBS1", "SBS5", "SBS10a", "SBS10b", "SBS17a", "SBS17b", "SBS18", "SBS40a", "SBS44"],
    # Flat, mutually similar signatures: hardest case for any factorisation.
    "hard_flat": ["SBS3", "SBS5", "SBS40a", "SBS8", "SBS39", "SBS89"],
}


def cosine_matrix(A, B):
    A = A / np.linalg.norm(A, axis=0, keepdims=True)
    B = B / np.linalg.norm(B, axis=0, keepdims=True)
    return A.T @ B


def choose_signatures(cosmic, args, rng):
    if args.signatures:
        names = args.signatures.split(",")
    elif args.preset:
        names = PRESETS[args.preset]
    else:
        pool = [s for s in cosmic.columns if args.include_artefacts or s not in ARTEFACTS]
        names = []
        # Greedy draw that rejects candidates too similar to an already-chosen signature.
        for s in rng.permutation(pool):
            if len(names) == args.n_signatures:
                break
            if names and cosine_matrix(cosmic[[s]].values, cosmic[names].values).max() > args.max_cosine:
                continue
            names.append(s)
        if len(names) < args.n_signatures:
            raise SystemExit(f"only found {len(names)} signatures with pairwise cosine <= {args.max_cosine}")
    missing = [s for s in names if s not in cosmic.columns]
    if missing:
        raise SystemExit(f"not in COSMIC file: {missing}")
    return names


def perturb(W, conc, rng):
    if conc <= 0:
        return W.copy()
    out = np.empty_like(W)
    for k in range(W.shape[1]):
        out[:, k] = rng.dirichlet(conc * W[:, k] + 1e-6)
    return out


def simulate(args):
    rng = np.random.default_rng(args.seed)
    cosmic = pd.read_csv(args.cosmic, sep="\t", index_col=0)
    names = choose_signatures(cosmic, args, rng)
    W_cosmic = cosmic[names].values
    W = perturb(W_cosmic, args.perturb_conc, rng)
    K, N = len(names), args.n_samples

    always = set(args.always_on.split(",")) & set(names) if args.always_on else set()
    H_prop = np.zeros((K, N))
    for j in range(N):
        active = np.array([n in always or rng.random() < args.p_active for n in names])
        if active.sum() < args.min_active:
            extra = rng.choice(np.flatnonzero(~active), args.min_active - active.sum(), replace=False)
            active[extra] = True
        H_prop[active, j] = rng.dirichlet(np.full(active.sum(), args.alpha))

    burden = rng.lognormal(np.log(args.median_burden), args.burden_sigma, N)
    H_counts = H_prop * burden
    mu = W @ H_counts
    if args.noise == "poisson":
        X = rng.poisson(mu)
    elif args.noise == "negbin":
        # numpy parametrisation: n=r, p=r/(r+mu)  ->  mean mu, var mu + mu^2/r
        X = rng.negative_binomial(args.nb_dispersion, args.nb_dispersion / (args.nb_dispersion + mu))
    else:
        X = np.rint(mu).astype(int)

    samples = [f"S{j + 1:04d}" for j in range(N)]
    os.makedirs(args.out, exist_ok=True)
    idx = pd.Index(cosmic.index, name="Type")
    pd.DataFrame(X, index=idx, columns=samples).to_csv(os.path.join(args.out, "counts.tsv"), sep="\t")
    pd.DataFrame(W, index=idx, columns=names).to_csv(os.path.join(args.out, "W_true.tsv"), sep="\t")
    pd.DataFrame(W_cosmic, index=idx, columns=names).to_csv(os.path.join(args.out, "W_cosmic.tsv"), sep="\t")
    sig_idx = pd.Index(names, name="Signature")
    pd.DataFrame(H_counts, index=sig_idx, columns=samples).to_csv(
        os.path.join(args.out, "H_true_counts.tsv"), sep="\t", float_format="%.4f")
    pd.DataFrame(H_prop, index=sig_idx, columns=samples).to_csv(
        os.path.join(args.out, "H_true_proportions.tsv"), sep="\t", float_format="%.6f")

    C = cosine_matrix(W, W)
    np.fill_diagonal(C, 0)
    params = vars(args) | {
        "signatures": names,
        "max_pairwise_cosine_true": float(C.max()),
        "mean_burden_observed": float(X.sum(0).mean()),
        "mean_active_per_sample": float((H_prop > 0).sum(0).mean()),
    }
    with open(os.path.join(args.out, "params.json"), "w") as f:
        json.dump(params, f, indent=2)
    return params


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--out", required=True)
    p.add_argument("--cosmic", default=DEFAULT_COSMIC)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--n-samples", type=int, default=200)
    g = p.add_argument_group("which signatures (pick one: --signatures, --preset, or random draw)")
    g.add_argument("--signatures", help="comma-separated COSMIC names, e.g. SBS1,SBS5,SBS13")
    g.add_argument("--preset", choices=sorted(PRESETS))
    g.add_argument("--n-signatures", type=int, default=8, help="size of random draw")
    g.add_argument("--max-cosine", type=float, default=0.8, help="max pairwise cosine in random draw")
    g.add_argument("--include-artefacts", action="store_true")
    g = p.add_argument_group("exposures")
    g.add_argument("--always-on", default="SBS1,SBS5", help="active in every sample (if selected)")
    g.add_argument("--p-active", type=float, default=0.4, help="prob. each other signature is active")
    g.add_argument("--min-active", type=int, default=1)
    g.add_argument("--alpha", type=float, default=1.0, help="Dirichlet concentration; <1 = one signature dominates")
    g.add_argument("--median-burden", type=float, default=2000, help="WGS ~1e3-1e4, WES ~50-300")
    g.add_argument("--burden-sigma", type=float, default=1.0, help="log-normal sd of burden")
    g = p.add_argument_group("noise")
    g.add_argument("--noise", choices=["poisson", "negbin", "none"], default="negbin")
    g.add_argument("--nb-dispersion", type=float, default=50.0, help="smaller = more overdispersed")
    g.add_argument("--perturb-conc", type=float, default=0.0,
                   help="Dirichlet concentration for perturbing signatures (0 = exact COSMIC; ~500 mild, ~100 strong)")
    args = p.parse_args()
    params = simulate(args)
    print(json.dumps({k: params[k] for k in ("out", "signatures", "max_pairwise_cosine_true",
                                             "mean_burden_observed", "mean_active_per_sample")}, indent=2))


if __name__ == "__main__":
    main()
